In [1]:
# Import statments

import h5py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import fcluster, dendrogram, linkage
from scipy.spatial.distance import squareform
import os
import re

In [ ]:
# curr_dir = os.getcwd()
# base_dir = os.path.dirname(curr_dir)
# data_dir = os.path.join(base_dir, 'Data')
# temp = os.path.join( data_dir , 'mitopark')
# filename = os.path.join( temp , 'session_1_out.mat') 

In [2]:
filename = (r'X:\Members\Mia-Sanjana-Hadent\Processed Data\042025_2mp\session_1_out.mat')

In [ ]:
# # If similarity matrix and idx files exist, extract and save them

# with h5py.File(filename, 'r') as file:
#     if 'Clusters' in file:
#         clusters_group = file['Clusters']
#         if 'sim' in clusters_group:
#             sim_data = clusters_group['sim'][:]
#             sim_matrix_data = pd.DataFrame(sim_data)

In [ ]:
# with h5py.File(filename, 'r') as file:
#     if 'Clusters' in file:
#         clusters_group = file['Clusters']

#         if 'idx' in clusters_group:
#             idx_data = clusters_group['idx'][:]
#             # clusters_data = pd.DataFrame(idx_data).transpose()

In [ ]:
# clusters_data = pd.DataFrame(idx_data).transpose()

In [3]:
with h5py.File(filename, 'r') as file:
    if 'Clusters' in file:
        clusters_group = file['Clusters']
        if 'sim' in clusters_group:
            # Do NOT read the entire dataset into memory.
            # Keep only small preview in memory and store metadata so you can stream later.
            sim_ds = clusters_group['sim']                       # h5py.Dataset (lazy, on-disk)
            print("sim dataset shape:", sim_ds.shape, "dtype:", sim_ds.dtype)

            # Detect channel axis: use first channel if present
            if sim_ds.ndim == 3:
                sim_channel = 0
            else:
                sim_channel = None

            N = sim_ds.shape[0]
            # preview size for plotting (adjust to taste; smaller = less memory)
            preview_size = min(1000, N)
            preview_idx = np.linspace(0, N - 1, preview_size).astype(int)

            # Stream only the rows/cols needed for the preview (read in small row-chunks)
            row_chunk = 200
            preview_blocks = []
            for start in range(0, len(preview_idx), row_chunk):
                r_idx = preview_idx[start:start + row_chunk]

                # h5py does not accept 2D index arrays (e.g. r_idx[:, None]) for fancy indexing.
                # Read requested columns per-row and stack to avoid creating 2D index arrays.
                if sim_channel is None:
                    block_rows = [np.array(sim_ds[int(r), preview_idx], dtype=np.float32) for r in r_idx]
                else:
                    block_rows = [np.array(sim_ds[int(r), preview_idx, sim_channel], dtype=np.float32) for r in r_idx]

                block = np.vstack(block_rows)                       # shape (len(r_idx), preview_size)
                preview_blocks.append(block)

            sim_matrix_preview = np.vstack(preview_blocks)         # shape (preview_size, preview_size)

            # Provide a small DataFrame for plotting & downstream preview operations
            sim_matrix_data = pd.DataFrame(sim_matrix_preview, index=preview_idx, columns=preview_idx)

            # Save metadata so other cells can reopen and stream as needed (do NOT rely on sim_ds after file closed)
            sim_shape = sim_ds.shape
            sim_dtype = sim_ds.dtype
            sim_preview_idx = preview_idx
            sim_preview_size = preview_size


sim dataset shape: (121067, 121067) dtype: float64


In [4]:
with h5py.File(filename, 'r') as file:
    if 'Clusters' in file:
        clusters_group = file['Clusters']

        if 'idx' in clusters_group:
            idx_ds = clusters_group['idx']
            print("idx dataset shape:", idx_ds.shape, "dtype:", idx_ds.dtype)

            # Read idx efficiently. For typical idx sizes this is cheap; if extremely large, read in chunks.
            N_idx = idx_ds.shape[0] if getattr(idx_ds, 'shape', None) else len(idx_ds)
            if N_idx <= 5_000_000:
                # single read for moderate sizes (keeps code simple)
                idx_arr = idx_ds[()].squeeze()
            else:
                # chunked read to avoid single huge allocation spike
                chunk = 500_000
                parts = []
                for start in range(0, N_idx, chunk):
                    parts.append(np.array(idx_ds[start:start+chunk], dtype=idx_ds.dtype))
                idx_arr = np.concatenate(parts)

            # Ensure 1D int array (matches prior behavior)
            idx_arr = np.asarray(idx_arr).ravel().astype(int, copy=False)

            # Create a small DataFrame with same orientation you used previously.
            # The original code did: clusters_data = pd.DataFrame(idx_data).transpose()
            # Keep that behavior but avoid unnecessary temporaries.
            idx_df = pd.DataFrame(idx_arr)          # shape (N, 1)


idx dataset shape: (1, 121067) dtype: float64


In [5]:
idx_df

,0
0,1
1,1
2,1
3,1
4,1
...,...
121062,96
121063,91
121064,96
121065,79


In [6]:
# n = int(3599904/300)

idx_df = pd.DataFrame(idx_arr)      # shape (N, 1)
idx_df.columns = ['cluster']        # only changes labels, avoids creating new DataFrame with copied data
clusters_data = idx_df

In [7]:
#Sort the idx file so that the timstamps are ordered by the cluster 

# clusters_data_sorted = clusters_data.sort_values(by='cluster')
# clusters_data_sorted['cluster'] = clusters_data_sorted['cluster'].astype( int )


cluster_vals = clusters_data['cluster'].to_numpy(dtype=int, copy=False)
sorted_order = np.argsort(cluster_vals, kind='stable')
sorted_indices = np.arange(cluster_vals.shape[0])[sorted_order]
clusters_data_sorted = clusters_data.iloc[sorted_order]
unique_clusters, counts = np.unique(cluster_vals, return_counts=True)
cluster_counts = pd.DataFrame({'cluster': unique_clusters, 'count': counts})
cluster_counts = cluster_counts.sort_values(by='cluster').reset_index(drop=True)

# sim_matrix_data = sim_matrix_data * -1

In [ ]:
# #  Plot given similarity matrix

# def plot_similarity_matrix(sim_matrix, title, xlabel, ylabel, colorbar_threshold=-0.3, save=False, path=None):
#     data_array = sim_matrix.to_numpy()
#     cluster_labels = sim_matrix.index.astype(str)  
    
#     plt.figure(figsize=(12, 9))
#     im = plt.imshow(data_array, cmap='Blues', interpolation='nearest', vmin=colorbar_threshold, vmax=0)
#     plt.colorbar(im)
#     plt.clim(colorbar_threshold, 0)
#     plt.title(title)
#     plt.xlabel(xlabel)
#     plt.ylabel(ylabel)

#     # Set cluster labels for axes
#     plt.xticks(ticks=np.arange(len(cluster_labels)), labels=cluster_labels, rotation=90)
#     plt.yticks(ticks=np.arange(len(cluster_labels)), labels=cluster_labels)

#     # Save if needed
#     if save:
#         if path is None:
#             safe_title = re.sub(r'[^\w\-_. ]', '_', title).replace(' ', '_')
#             path = f"{safe_title}.eps"

#         eps_dir = os.path.dirname(path)
#         if eps_dir:
#             os.makedirs(eps_dir, exist_ok=True)

#         base_filename = os.path.splitext(os.path.basename(path))[0]
#         eps_path = os.path.join(eps_dir, base_filename + '.eps')
#         plt.savefig(eps_path, format='eps', bbox_inches='tight')
#         print(f"EPS saved to: {eps_path}")

#         parent_dir = os.path.dirname(eps_dir)
#         png_dir = os.path.join(parent_dir, 'png')
#         os.makedirs(png_dir, exist_ok=True)

#         png_path = os.path.join(png_dir, base_filename + '.png')
#         plt.savefig(png_path, format='png', dpi=300, bbox_inches='tight')
#         print(f"PNG saved to: {png_path}")

#     plt.tight_layout()
#     plt.show()

In [8]:
def plot_similarity_matrix(sim_matrix, title, xlabel, ylabel, colorbar_threshold=-0.3, save=False, path=None):
    """
    Safe plotting helper:
      - accepts small pd.DataFrame or small numpy array
      - avoids forcing huge DataFrame -> huge numpy copies
      - only creates axis labels if matrix is reasonably small
    """
    if isinstance(sim_matrix, pd.DataFrame):
        data_array = sim_matrix.values  # OK for small previews
        labels = sim_matrix.index.astype(str)
    else:
        data_array = np.asarray(sim_matrix, dtype=np.float32)
        labels = None

    plt.figure(figsize=(12, 9))
    im = plt.imshow(data_array, cmap='Blues', interpolation='nearest', vmin=colorbar_threshold, vmax=0)
    plt.colorbar(im)
    plt.clim(colorbar_threshold, 0)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)

    # Only set tick labels when matrix is small to avoid huge memory/time overhead
    if labels is not None and data_array.shape[0] <= 2000:
        plt.xticks(ticks=np.arange(len(labels)), labels=labels, rotation=90)
        plt.yticks(ticks=np.arange(len(labels)), labels=labels)

    if save:
        if path is None:
            safe_title = re.sub(r'[^\\w\\-_. ]', '_', title).replace(' ', '_')
            path = f"{safe_title}.eps"
        eps_dir = os.path.dirname(path)
        if eps_dir:
            os.makedirs(eps_dir, exist_ok=True)
        base_filename = os.path.splitext(os.path.basename(path))[0]
        eps_path = os.path.join(eps_dir, base_filename + '.eps')
        plt.savefig(eps_path, format='eps', bbox_inches='tight')
        print(f"EPS saved to: {eps_path}")
        parent_dir = os.path.dirname(eps_dir)
        png_dir = os.path.join(parent_dir, 'png')
        os.makedirs(png_dir, exist_ok=True)
        png_path = os.path.join(png_dir, base_filename + '.png')
        plt.savefig(png_path, format='png', dpi=300, bbox_inches='tight')
        print(f"PNG saved to: {png_path}")

    plt.tight_layout()
    plt.show()

In [ ]:
# Plots raw similarity matrix

plt.figure(figsize=(12, 9))
im = plt.imshow(sim_matrix_data.to_numpy(), cmap='Blues', interpolation='nearest', vmin=-0.30, vmax=0)
plt.colorbar(im)
plt.clim(-0.30, 0)
plt.title("Similarity Matrix (Raw)")
plt.xlabel("Time Segment X")
plt.ylabel("Time Segment Y")

plt.show()


In [ ]:
# Reorders original similarity matrix by ordering by cluster label instead of timestamp


# sorted_indices = clusters_data_sorted.index
# reordered_matrix = sim_matrix_data.loc[sorted_indices, sorted_indices]

In [9]:
# Reorders original similarity matrix by ordering by cluster label instead of timestamp
# Note: sim_matrix_data is a small preview (sim_preview_idx). Do NOT attempt to build full N×N reordered matrix in memory.

# Use the already computed sorted_indices (full range). Intersect with preview indices to build a preview-only reordered matrix.
_preview_idx = globals().get('sim_preview_idx', globals().get('preview_idx', None))
if _preview_idx is None:
    raise RuntimeError("Preview indices not found (sim_preview_idx / preview_idx). Run the sim preview cell first.")

# indices from the full sorted order that are present in the preview
preview_sorted_idx = np.intersect1d(sorted_indices, _preview_idx, assume_unique=False)

if len(preview_sorted_idx) == 0:
    # nothing in common (unlikely) — fall back to the original preview
    reordered_matrix = sim_matrix_data.copy()
else:
    # Stream the small reordered-preview directly from disk (reopen file for safe access)
    row_chunk = 200
    preview_blocks = []
    with h5py.File(filename, 'r') as f:
        ds = f['Clusters']['sim']
        # if dataset has 3 dims, use channel 0 by default (mirrors earlier logic)
        has_channel = (ds.ndim == 3)
        channel = 0 if has_channel else None

        for start in range(0, len(preview_sorted_idx), row_chunk):
            r_idx = preview_sorted_idx[start:start + row_chunk]
            # h5py fancy 2D indexing may be restricted; read per-row and stack to keep memory low
            if channel is None:
                block_rows = [np.array(ds[int(r), preview_sorted_idx], dtype=np.float32) for r in r_idx]
            else:
                block_rows = [np.array(ds[int(r), preview_sorted_idx, channel], dtype=np.float32) for r in r_idx]
            preview_blocks.append(np.vstack(block_rows))

    reordered_preview = np.vstack(preview_blocks)
    reordered_matrix = pd.DataFrame(reordered_preview, index=preview_sorted_idx, columns=preview_sorted_idx)

# Reminder: reordered_matrix here is a preview (subsample). To compute a full reordered view or cluster-averaged final_matrix,
# stream blocks from the HDF5 file per-cluster (don't create full N×N in memory).

In [ ]:
#Plots similarity matrix reordered by cluster value --> all of the times with cluster 1 are at the beginning, then 2, etc

# NOTE : if the ordering of the clusters doesn't differ much from orginal to reordered, this matrix will look very similar to the previous one

plt.figure(figsize=(12, 9))
im = plt.imshow(reordered_matrix.to_numpy(), cmap='Blues', interpolation='nearest', vmin=-0.30, vmax=0)
plt.colorbar(im)
plt.clim(-0.30, 0)
plt.title("Similarity Matrix (Reordered)")
plt.xlabel("Time Segment X")
plt.ylabel("Time Segment Y")

plt.show()

In [ ]:
# # Figures out how many rows of data exist for each cluster label

# cluster_counts = clusters_data_sorted['cluster'].value_counts().reset_index()
# cluster_counts.columns = ['cluster', 'count']
# cluster_counts = cluster_counts.sort_values(by='cluster')


In [ ]:
# # Computes a condensed version of original matrix by averaging together the rows and columns where the cluster label is the same

# clusters = cluster_counts['cluster']
# counts = cluster_counts['count'].values

# final_matrix = np.zeros((len(clusters), len(clusters)))

# # Compute averages for each cluster pair
# for i, cluster_i in enumerate(clusters):
#     for j, cluster_j in enumerate(clusters):
#         # Get the indices for rows and columns in the original matrix for the respective clusters
#         rows_i = clusters_data_sorted[clusters_data_sorted['cluster'] == cluster_i].index
#         rows_j = clusters_data_sorted[clusters_data_sorted['cluster'] == cluster_j].index
        
#         # Compute the average of the submatrix
#         submatrix = reordered_matrix.loc[rows_i, rows_j]
#         # submatrix = sim_matrix_data.loc[rows_i, rows_j]
#         final_matrix[i, j] = submatrix.values.mean()

# final_matrix_df = pd.DataFrame(final_matrix, index=clusters, columns=clusters)


In [10]:
# Compute cluster-averaged condensed matrix by streaming from the HDF5 sim dataset.
# Assumes: clusters_data_sorted (DataFrame with 'cluster') and cluster_counts exist.
clusters = cluster_counts['cluster'].to_numpy()
K = len(clusters)

# Build list of original indices for each cluster (numpy arrays)
cluster_indices = [clusters_data_sorted[clusters_data_sorted['cluster'] == c].index.to_numpy() for c in clusters]

final_matrix = np.zeros((K, K), dtype=np.float64)

# Tune row_chunk depending on memory / I/O - larger = fewer Python calls but larger temp arrays
row_chunk = 500

with h5py.File(filename, 'r') as f:
    ds = f['Clusters']['sim']
    has_channel = (ds.ndim == 3)
    channel = 0 if has_channel else None

    for i in range(K):
        rows_i = cluster_indices[i]
        if rows_i.size == 0:
            # keep zeros or set nan if preferred
            final_matrix[i, :] = np.nan
            final_matrix[:, i] = np.nan
            continue

        for j in range(i, K):
            cols_j = cluster_indices[j]
            if cols_j.size == 0:
                final_matrix[i, j] = np.nan
                final_matrix[j, i] = np.nan
                continue

            total = 0.0
            count = 0

            # Read rows_i in small chunks to avoid large allocations
            for start in range(0, len(rows_i), row_chunk):
                r_chunk = rows_i[start:start + row_chunk]

                # Read each row in the chunk for the selected columns and stack
                if channel is None:
                    block_rows = [np.asarray(ds[int(r), cols_j], dtype=np.float64) for r in r_chunk]
                else:
                    block_rows = [np.asarray(ds[int(r), cols_j, channel], dtype=np.float64) for r in r_chunk]

                if len(block_rows) == 0:
                    continue

                block = np.vstack(block_rows)  # shape (len(r_chunk), len(cols_j))
                total += block.sum()
                count += block.size

            # mean similarity between cluster i and cluster j
            mean_val = (total / count) if count > 0 else np.nan
            final_matrix[i, j] = mean_val
            final_matrix[j, i] = mean_val  # symmetry

# Small K x K DataFrame for downstream use
final_matrix_df = pd.DataFrame(final_matrix, index=clusters, columns=clusters)

KeyboardInterrupt: 

In [ ]:
# Ensures given similarity matrix has correct dendrogram, creates linkage matrix and plots dendrogram

def create_dendrogram( similarity_matrix , title , max_height = 0.1 , save = False , path = None ):
    final_matrix_df = (similarity_matrix + similarity_matrix.T) / 2
    final_matrix_df = final_matrix_df.abs()
    np.fill_diagonal(final_matrix_df.values, 0)
    condensed_matrix = squareform(final_matrix_df)
    linkage_matrix = linkage(condensed_matrix, method='average')

    # metric options are ‘braycurtis’, ‘canberra’, ‘chebyshev’, ‘cityblock’, ‘correlation’, ‘cosine’, ‘dice’, ‘euclidean’, ‘hamming’, ‘jaccard’, ‘jensenshannon’, 
    # ‘kulczynski1’, ‘mahalanobis’, ‘matching’, ‘minkowski’, ‘rogerstanimoto’, ‘russellrao’, ‘seuclidean’, ‘sokalmichener’, ‘sokalsneath’, ‘sqeuclidean’, ‘yule’
    # linkage_matrix = linkage(condensed_matrix, method='average' , metric='euclidean')

    # Plot the dendrogram
    plt.figure(figsize=(10, 7))
    labels = [str(i + 1) for i in range(similarity_matrix.shape[0])]
    dendrogram(linkage_matrix, labels=labels, leaf_rotation=90, leaf_font_size=10)
    # dendrogram(linkage_matrix, leaf_rotation=90, leaf_font_size=10)
    plt.title(title)
    plt.xlabel("Clusters")
    plt.ylabel("Distance")
    plt.ylim(0, max_height)
    plt.tight_layout()

    if save:
        if path is None:
            safe_title = re.sub(r'[^\w\-_. ]', '_', title).replace(' ', '_')
            path = f"{safe_title}.eps"

        eps_dir = os.path.dirname(path)
        if eps_dir:
            os.makedirs(eps_dir, exist_ok=True)

        base_filename = os.path.splitext(os.path.basename(path))[0]
        eps_path = os.path.join(eps_dir, base_filename + '.eps')
        plt.savefig(eps_path, format='eps', bbox_inches='tight')
        print(f"EPS saved to: {eps_path}")

        parent_dir = os.path.dirname(eps_dir)
        png_dir = os.path.join(parent_dir, 'png')
        os.makedirs(png_dir, exist_ok=True)

        png_path = os.path.join(png_dir, base_filename + '.png')
        plt.savefig(png_path, format='png', dpi=300, bbox_inches='tight')
        print(f"PNG saved to: {png_path}")


    plt.show()

    return linkage_matrix

In [ ]:
# save_path = os.path.join( poster_visualizations_dir , 'full_data\eps\dendrogram.eps')
full_linkage_matrix = create_dendrogram( final_matrix_df , "Dendrogram" , max_height = 0.11 , save = False )

In [ ]:
# Plot the dendrogram with a horizontal threshold line

def plot_dendrogram_with_threshold_line(linkage_matrix , threshold):
    plt.figure(figsize=(12, 7))
    labels = [str(i + 1) for i in range(final_matrix_df.shape[0])]
    dendrogram(linkage_matrix, labels = labels, leaf_rotation=90, leaf_font_size=10)
    plt.axhline(y=threshold, color='r', linestyle='--')  # Draw red line at threshold
    plt.title("Dendrogram with Threshold Line")
    plt.xlabel("Clusters")
    plt.ylabel("Distance")
    plt.show()

In [ ]:
plot_dendrogram_with_threshold_line( full_linkage_matrix , 0.015 )

In [ ]:
# Figures out which clusters should be grouped together based on a given threshold

def find_cluster_groups( linkage_matrix , sim_matrix , threshold):
    cluster_labels = fcluster(linkage_matrix, threshold, criterion='distance')
    cluster_groups_df = pd.DataFrame({'Cluster': sim_matrix.index - 1, 'Group': cluster_labels})
    cluster_groups_df['Cluster'] = cluster_groups_df['Cluster'] + 1
    clusters_groups_sorted = cluster_groups_df.sort_values(by='Group')
    return clusters_groups_sorted

In [ ]:
# clusters_groups_sorted = cluster_groups_df.sort_values(by='Group')
clusters_groups_sorted = find_cluster_groups( full_linkage_matrix , final_matrix_df , 0.015 )
clusters_groups_sorted
clusters_groups_sorted.to_csv(os.path.join( temp , 'groupedClusters.csv') ) 